### Observables

Wir nennen ein Objekt `obj` ein sog. **Observable**, wenn es eine Methode zur
Verfügung stellt, die es erlaubt, eine Funktion  als
Callback zu registrieren, und
diese Funktion dann beim Eintreffen  bestimmter Events
mit entsprechenden Argumenten aufruft.

Ein typisches Beispiel ist das Selection-Widget:
- Die Methode `observe(f, names='value')`, erlaubt eine Funktion `f` als Callback zu registrieren,
  welche bei einer Änderung des Attributs `value` mit  `f(change)` aufgerufen wird.
 Das Argument `change` hat u.a. Atrribute `change.old` und `change.new`, die den alten und neuen Wert des Attributs `value` enthalten.



Von der Idee her sind auch unsere biherigen Game-Komponenten *Observables*.
Die Funktion `game.update` wird jeweils von einer View-Komponente 
überschrieben (View registiert Callback).
Die Game-Komponente ruft dann diese Funktion mit Argumenten `event` und `data` auf, 
wenn etwas Darstellungsrelevantes passiert.  

Wir wollen unsere Spiele als **Observables** implementieren!

In [ ]:
# Select-Widget: typisches Observable

import widget_helpers as W
from IPython.display import display
from ipywidgets import Select

out = W.get_out()
select = Select(options=("A", "B"), value=None, rows=2)


@out.capture(clear_output=True)
def on_change(change):
    print(f"Der Wert das Attributes 'value' ist nun {change.new} (vorher {change.old})")


select.observe(on_change, names="value")
display(select, out)

In [ ]:
select.value

In [ ]:
select.value = "B"

***
**Klasse `Game` als Observable**

Der Spieler muss alle Felder einer 4x4 Grid mit Kreisen füllen.
Kreise können nicht auf der Diagonale platziert werden, sondern müssen dorthin verschoben werden.
- `place(pos)` setzt eine Kreis an Position `pos`,
- `move(old_pos, new_pos)` verschiebt einen Kreis.

Siehe `game1.py`

In [ ]:
from game1 import Game


def callback(event, data):
    print(f'event={event}, data={data}')


def test_game(game):
    game.new_game()
    for i in range(4):
        old_pos = (1, 0)
        new_pos = (i, i)
        game.place(old_pos)
        game.move(old_pos, new_pos)

    for i in range(16):
        row, col = divmod(i, 4)
        game.place((col, row))

    return game

In [ ]:
game = Game()
game.register_callback(callback)
game = test_game(game)

***
**Relevanten Code auslagern**  
Um nicht jedesmal den Code zum Erstellen der Liste `callbacks` und zum Registrieren und
Aufrufen der Callbacks in eine Game-Klasse einfügen zu müssen, lagern wird diesen Code in eine
Klasse `Observable` aus, von der dann jede Game-Klasse erbt.

In [ ]:
class Observable:
    def __init__(self):
        self.callbacks = []

    def register_callback(self, callback):
        self.callbacks.append(callback)

    def notify(self, event, data=None):
        for f in self.callbacks:
            f(event, data)


class Game(Observable):
    def __init__(self):
        super().__init__()
        self.ncol = 4
        self.nrow = 4
        self.placed = set()
        self.blocked = {(i, i) for i in range(4)}

    def new_game(self):
        self.placed.clear()
        self.success = False
        self.notify('new_game')

    def is_inside(self, col, row):
        return 0 <= col < self.ncol and 0 <= row < self.nrow

    def place(self, pos):
        data = None
        if self.is_inside(*pos) and pos not in self.blocked | self.placed:
            self.placed.add(pos)
            self.success = len(self.placed) == self.ncol * self.nrow
            data = pos, self.success

        self.notify('place', data)

    def move(self, old_pos, new_pos):
        data = None
        if self.is_inside(*new_pos) and old_pos in self.placed and new_pos not in self.placed:
            self.placed.remove(old_pos)
            self.placed.add(new_pos)
            data = old_pos, new_pos

        self.notify('move', data)

In [ ]:
game = Game()
game.register_callback(callback)
game = test_game(game)

### Observable ohne `__init__`
Wir möchten keine  `__init__`-Methode in der Klasse `Observable`, damit
die erbende Klasse diese Methode nicht mit `super().__init__()` aufrufen muss.

Die  `__init__`-Methode erstellt die Liste `callbacks`.
Existiert diese Liste nicht, erzeugt `callbacks.append(callback)` in der Methode `register_callback` einen Fehler.  

**Lösung**: Wir prüfen vor dem Anhängen, ob die Liste `callbacks` existiert und erstellen sie gegebenenfalls.

- `hasattr(obj, name)` gibt `True` zurück, falls das Objekt `obj` ein Attribute mit Namen `name` hat.
   z.B. `hasattr(observable, 'callbacks`)`

- `setattr(obj, name, value)` weist dem Attribute `name` von `obj` den Wert `value` zu.  
  `setattr(observable, 'callbacks, [])` entspricht `observable.callbacks = []`

- `getattr(object, name)` gibt den Wert des Attributs `name` zurück. Erzeugt einen `AttributeError` falls `object` kein solches
Attribute hat.

- `getattr(object, name, default)` gibt  `default` zurück, falls `object` kein solches
Attribute hat.

In [ ]:
class Observable:
    def register_callback(self, callback):
        if not hasattr(self, 'callbacks'):
            self.callbacks = []

        if callback not in self.callbacks:
            self.callbacks.append(callback)

    def _notify(self, event, data=None):
        '''calls all registered callbacks with the arguments event and data'''

        for callback in getattr(self, 'callbacks', []):
            callback(event, data)


class A(Observable):
    pass

In [ ]:
a = A()
hasattr(a, 'callbacks')

In [ ]:
getattr(a, 'callbacks', [])

In [ ]:
a.register_callback(print)  # nun wird die Liste callbacks erstellt
hasattr(a, 'callbacks')

In [ ]:
a.callbacks

In [ ]:
from functools import wraps


def notify(f):
    @wraps(f)
    def wrapper(self, *args, **kwargs):
        data = f(self, *args, **kwargs)
        self._notify(f.__name__, data)
    return wrapper


class Game(Observable):
    def __init__(self):
        self.ncol = 4
        self.nrow = 4
        self.placed = set()
        self.blocked = {(i, i) for i in range(4)}

    @notify  # ruft _notify(<methodenname>, <zurueckgeg. Argumente>) auf
    def new_game(self):
        self.placed.clear()
        self.success = False

    def is_inside(self, col, row):
        return 0 <= col < self.ncol and 0 <= row < self.nrow

    @notify
    def place(self, pos):
        if self.is_inside(*pos) and pos not in self.blocked | self.placed:
            self.placed.add(pos)
            self.success = len(self.placed) == self.ncol * self.nrow
            return pos, self.success

    @notify
    def move(self, old_pos, new_pos):
        if self.is_inside(*new_pos) and old_pos in self.placed and new_pos not in self.placed:
            self.placed.remove(old_pos)
            self.placed.add(new_pos)
            return old_pos, new_pos

In [ ]:
game = Game()
game.register_callback(callback)
game = test_game(game)